<a href="https://colab.research.google.com/github/GitHubAman2004/stock_price_predictor/blob/main/analysis_extended_dataset_for_stock_price_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install yfinance pandas

In [ ]:
%pip install scikit-learn

In [ ]:
import torch
dummy_x=torch.randn(32,60,5)

In [ ]:
import torch
import torch.nn as nn
cnn_input=dummy_x.permute(0,2,1)
conv_layer=nn.Conv1d(in_channels=5,out_channels=16,kernel_size=3,padding=1)
cnn_output=conv_layer(cnn_input)
final_output=cnn_output.permute(0,2,1)
print("original shape", dummy_x.shape)
print("after cnn block", final_output.shape)

original shape torch.Size([32, 60, 5])
after cnn block torch.Size([32, 60, 16])


In [ ]:
import torch
import torch.nn as nn
lstm_layer=nn.LSTM(input_size=16,hidden_size=16,batch_first=True)
lstm_output,(h_n,c_n)=lstm_layer(final_output)
print("shape entering lstm",final_output.shape)
print("shape exiting lstm",lstm_output.shape)

shape entering lstm torch.Size([32, 60, 16])
shape exiting lstm torch.Size([32, 60, 16])


In [ ]:
import torch
import torch.nn as nn
attention_layer=nn.MultiheadAttention(embed_dim=32,num_heads=4,batch_first=True)

In [ ]:
import torch
import torch.nn as nn

class HybridForecastingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Conv1d(in_channels=9, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
        self.fc = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.cnn(x))
        x = self.dropout(x)
        x = x.permute(0, 2, 1)

        x, _ = self.lstm(x)
        x, _ = self.attention(x, x, x)

        last_day_features = x[:, -1, :]
        prediction = self.fc(last_day_features)

        return prediction

model = HybridForecastingModel()
dummy_x = torch.randn(32, 60, 9)
final_prediction = model(dummy_x)

print("Final prediction shape:", final_prediction.shape)

Final prediction shape: torch.Size([32, 1])


In [ ]:
import yfinance as yf
import pandas as pd
ticker="AAPL"
print("Downloading data from {ticker}...")
df=yf.download(ticker,period='max')
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df['Daily_Return'] = df['Close'].pct_change()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['Volatility'] = df['Daily_Return'].rolling(window=10).std()
df = df.dropna()
df = df.tail(10000)
print("\n--- Data successfully downloaded and upgraded! ---")
print("Total features:", len(df.columns))
print(df.head(5))
print("\n--- Data successfully downloaded! ---")
print("Total trading days:", len(df))
print("\nFirst 3 days of data:")
print(df.head(5))

/tmp/ipykernel_782/4030198388.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df=yf.download(ticker,period='max')
[*********************100%***********************]  1 of 1 completed


--- Data successfully downloaded and upgraded! ---
Total features: 9
Price           Open      High       Low     Close     Volume Daily_Return  \
Ticker          AAPL      AAPL      AAPL      AAPL       AAPL                
Date                                                                         
1986-10-13  0.113151  0.118276  0.112724  0.118276   99680000     0.041350   
1986-10-14  0.118276  0.120411  0.115287  0.116141  199337600    -0.018048   
1986-10-15  0.114433  0.114433  0.111871  0.114006  205408000    -0.018381   
1986-10-16  0.114006  0.115714  0.113579  0.114860  135766400     0.007490   
1986-10-17  0.115287  0.116141  0.114006  0.114860  151872000     0.000000   

Price         SMA_10    SMA_50 Volatility  
Ticker                                     
Date                                       
1986-10-13  0.114860  0.117592   0.021290  
1986-10-14  0.115031  0.117763   0.020710  
1986-10-15  0.114774  0.117849   0.020647  
1986-10-16  0.114604  0.118019   0.020864

In [ ]:
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
sequence_length = 60
X = []
y = []
for i in range(sequence_length, len(scaled_data) - 1):
    X.append(scaled_data[i - sequence_length : i])
    today_close = scaled_data[i, 3]
    tomorrow_close = scaled_data[i + 1, 3]
    price_change = (tomorrow_close - today_close) / (today_close + 1e-7)
    if price_change > 0.002:
        y.append(1)
    else:
        y.append(0)
X_tensor = torch.tensor(np.array(X), dtype=torch.float32)
y_tensor = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(1)

print("Dataset successfully refined!")
print("X shape (Input):", X_tensor.shape)
print("y shape (Target):", y_tensor.shape)

Dataset successfully refined!
X shape (Input): torch.Size([9939, 60, 9])
y shape (Target): torch.Size([9939, 1])


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
split_idx=int(len(X_tensor)*0.80)

X_train=X_tensor[:split_idx]
y_train=y_tensor[:split_idx]

X_test=X_tensor[split_idx:]
y_test=y_tensor[split_idx:]

train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)

batch_size=32

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=False)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

print("--- Data perfectly partitioned! ---")
print(f"Training samples: {len(X_train)} (Split into {len(train_loader)} batches of {batch_size})")
print(f"Testing samples: {len(X_test)} (Split into {len(test_loader)} batches of {batch_size})")

--- Data perfectly partitioned! ---
Training samples: 7951 (Split into 249 batches of 32)
Testing samples: 1988 (Split into 63 batches of 32)


In [ ]:
import torch.optim as optim
import torch.nn as nn
total_samples=len(train_loader.dataset)
neg_count=0
pos_count=0
for _,batch_y in train_loader:
  pos_count+=(batch_y==1).sum().item()
  neg_count+=(batch_y==0).sum().item()
print(f"Positive samples: {pos_count}")
print(f"Negative samples: {neg_count}")
print(f"pos_weight value: {neg_count / pos_count:.2f}")
model=HybridForecastingModel()
pos_weight=torch.tensor([neg_count/pos_count])
criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer=optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-5)

epochs=10
print("Starting Advanced Training")

for epoch in range (epochs):
  total_tp=0;
  total_fp=0;
  total_tn=0;
  total_fn=0;
  running_loss=0.0
  for batch_X,batch_y in train_loader:
     optimizer.zero_grad()
     predictions=model(batch_X)
     loss=criterion(predictions,batch_y)
     loss.backward()
     optimizer.step()

     running_loss+=loss.item()
     predicted_classes=(predictions>0).float()


     total_tp += ((predicted_classes == 1) & (batch_y == 1)).sum().item()
     total_fp += ((predicted_classes == 1) & (batch_y == 0)).sum().item()
     total_fn += ((predicted_classes == 0) & (batch_y == 1)).sum().item()
     total_tn += ((predicted_classes == 0) & (batch_y == 0)).sum().item()

  epoch_loss=running_loss/len(train_loader)

  total_samples = total_tp + total_tn + total_fp + total_fn
  accuracy = (total_tp + total_tn) / total_samples

  precision = total_tp / (total_tp + total_fp + 1e-7)

  recall = total_tp / (total_tp + total_fn + 1e-7)

  f1_score = 2 * (precision * recall) / (precision + recall + 1e-7)

  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {accuracy*100:.1f}% | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1_score:.3f}")

print("--- Training Complete! ---")

Positive samples: 3757
Negative samples: 4194
pos_weight value: 1.12
Starting Advanced Training
Epoch [1/10] | Loss: 0.7317 | Acc: 49.6% | Precision: 0.472 | Recall: 0.550 | F1: 0.508
Epoch [2/10] | Loss: 0.7315 | Acc: 51.2% | Precision: 0.476 | Recall: 0.337 | F1: 0.395
Epoch [3/10] | Loss: 0.7315 | Acc: 51.0% | Precision: 0.474 | Recall: 0.335 | F1: 0.392
Epoch [4/10] | Loss: 0.7315 | Acc: 50.8% | Precision: 0.472 | Recall: 0.338 | F1: 0.393
Epoch [5/10] | Loss: 0.7314 | Acc: 50.8% | Precision: 0.472 | Recall: 0.338 | F1: 0.394
Epoch [6/10] | Loss: 0.7314 | Acc: 50.9% | Precision: 0.473 | Recall: 0.340 | F1: 0.395
Epoch [7/10] | Loss: 0.7314 | Acc: 50.8% | Precision: 0.471 | Recall: 0.337 | F1: 0.393
Epoch [8/10] | Loss: 0.7314 | Acc: 50.9% | Precision: 0.473 | Recall: 0.331 | F1: 0.389
Epoch [9/10] | Loss: 0.7314 | Acc: 51.4% | Precision: 0.478 | Recall: 0.311 | F1: 0.377
Epoch [10/10] | Loss: 0.7313 | Acc: 50.4% | Precision: 0.472 | Recall: 0.412 | F1: 0.440
--- Training Complete! 